# KV Cache Benchmark Visualization

This notebook loads `benchmark_results.csv` and generates easy-to-read charts comparing generation with and without KV cache.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

CSV_PATH = Path('benchmark_results.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Could not find {CSV_PATH.resolve()}')

df = pd.read_csv(CSV_PATH)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], errors='coerce')
df['execution_device'] = df['execution_device'].astype(str)
df['mode'] = df['mode'].astype(str)

numeric_cols = [
    'prompt_tokens', 'new_tokens', 'total_tokens_processed', 'avg_seconds',
    'tokens_per_second', 'speedup_vs_no_cache', 'block_size', 'total_blocks'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Rows:', len(df))
print('Modes:', sorted(df['mode'].dropna().unique().tolist()))
print('Devices:', sorted(df['execution_device'].dropna().unique().tolist()))
display(df.head(5))

In [ ]:
# Build paired trial table (one row per timestamp/prompt/new_tokens config)
pair_keys = ['timestamp_utc', 'prompt', 'new_tokens', 'prompt_tokens', 'block_size', 'total_blocks', 'execution_device']

kv = (
    df[df['mode'] == 'kv_cache'][pair_keys + ['avg_seconds', 'tokens_per_second', 'speedup_vs_no_cache']]
    .rename(columns={
        'avg_seconds': 'kv_avg_seconds',
        'tokens_per_second': 'kv_tokens_per_second',
        'speedup_vs_no_cache': 'kv_reported_speedup'
    })
)

no_kv = (
    df[df['mode'] == 'no_kv_cache'][pair_keys + ['avg_seconds', 'tokens_per_second']]
    .rename(columns={
        'avg_seconds': 'no_kv_avg_seconds',
        'tokens_per_second': 'no_kv_tokens_per_second'
    })
)

paired = pd.merge(kv, no_kv, on=pair_keys, how='inner')
paired['computed_speedup'] = paired['no_kv_avg_seconds'] / paired['kv_avg_seconds']
paired = paired.sort_values(['new_tokens', 'prompt_tokens']).reset_index(drop=True)

print('Paired trials:', len(paired))
display(paired.head(10))

In [ ]:
# Chart 1: runtime vs generated tokens (both modes)
fig, ax = plt.subplots(figsize=(10, 6))
for mode, label in [('kv_cache', 'KV cache'), ('no_kv_cache', 'No KV cache')]:
    view = df[df['mode'] == mode].sort_values('new_tokens')
    ax.plot(view['new_tokens'], view['avg_seconds'], marker='o', linewidth=2, label=label)

ax.set_title('Average Runtime vs Generated Tokens')
ax.set_xlabel('new_tokens')
ax.set_ylabel('avg_seconds')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Chart 2: throughput comparison (tokens/sec)
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(paired))
ax.bar([i - 0.2 for i in x], paired['kv_tokens_per_second'], width=0.4, label='KV cache')
ax.bar([i + 0.2 for i in x], paired['no_kv_tokens_per_second'], width=0.4, label='No KV cache')

labels = [f"{int(n)} tok" for n in paired['new_tokens']]
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=60, ha='right')
ax.set_title('Throughput by Trial (tokens/sec)')
ax.set_xlabel('Trial (by new_tokens)')
ax.set_ylabel('tokens_per_second')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: speedup and summary
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(paired)), paired['computed_speedup'], color='steelblue')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax.set_xticks(range(len(paired)))
ax.set_xticklabels([f"{int(n)}" for n in paired['new_tokens']], rotation=60, ha='right')
ax.set_title('KV Cache Speedup by Trial')
ax.set_xlabel('new_tokens')
ax.set_ylabel('speedup (no_kv_avg_seconds / kv_avg_seconds)')
ax.grid(True, axis='y', alpha=0.3)

for i, b in enumerate(bars):
    value = paired['computed_speedup'].iloc[i]
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{value:.2f}x", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

summary = {
    'trials': int(len(paired)),
    'device': paired['execution_device'].mode().iloc[0] if len(paired) else 'unknown',
    'avg_speedup_x': float(paired['computed_speedup'].mean()) if len(paired) else float('nan'),
    'median_speedup_x': float(paired['computed_speedup'].median()) if len(paired) else float('nan'),
    'min_speedup_x': float(paired['computed_speedup'].min()) if len(paired) else float('nan'),
    'max_speedup_x': float(paired['computed_speedup'].max()) if len(paired) else float('nan'),
}
summary